# JetRacer Final — YOLOv8 Lane Following

Notebook này dùng YOLOv8 fine-tuned để detect **apex** (điểm đích của lane), sau đó đổi tọa độ x của apex thành steering. Chạy cell training một lần trước khi chạy camera.

In [ ]:
# Chạy một lần để tạo labels YOLO từ old_codes/road_following_A/apex và fine-tune.
# Lưu ý: venv phải có package ultralytics tương thích trước khi chạy.
from lanedetectionv3 import prepare_dataset, train, BEST_WEIGHTS
print(prepare_dataset())
# train(epochs=80, batch=8, imgsz=224, device=0)
print('Weights inference:', BEST_WEIGHTS)

In [ ]:
import os
import time
import numpy as np
import ipywidgets
from IPython.display import display
from jetcam.csi_camera import CSICamera
from jetcam.utils import bgr8_to_jpeg
from basic_motion import JetRacerController
from lanedetectionv3 import YoloLaneDetector, BEST_WEIGHTS

if not BEST_WEIGHTS.exists():
    raise FileNotFoundError('Chưa có YOLO fine-tuned weights. Bỏ comment train(...) ở cell trước và chạy xong trước.')

os.system('echo "jetson" | sudo -S systemctl restart nvargus-daemon')
time.sleep(2)
try:
    camera.running = False
    camera.unobserve_all()
except Exception:
    pass

camera = CSICamera(width=224, height=224, capture_fps=0)
car = JetRacerController()
detector = YoloLaneDetector(BEST_WEIGHTS, confidence=0.25, smoothing=0.35)
print('YOLOv8 apex detector loaded')

In [ ]:
state_widget = ipywidgets.ToggleButtons(options=['stop', 'live'], description='State', value='stop')
raw_widget = ipywidgets.Image(format='jpeg', width=224, height=224)
debug_widget = ipywidgets.Image(format='jpeg', width=224, height=224)
status_widget = ipywidgets.HTML(value='<b>YOLO:</b> ready')
steering_gain_slider = ipywidgets.FloatSlider(description='Steering Gain', min=0.0, max=2.0, value=1.0, step=0.05)
throttle_slider = ipywidgets.FloatSlider(description='Throttle', min=0.0, max=0.5, value=0.15, step=0.01)
blank = bgr8_to_jpeg(np.zeros((224, 224, 3), dtype=np.uint8))
raw_widget.value = blank
debug_widget.value = blank
display(ipywidgets.VBox([ipywidgets.HBox([raw_widget, debug_widget]), status_widget, steering_gain_slider, throttle_slider, state_widget]))

In [ ]:
def live_update(change):
    image = change['new']
    raw_widget.value = bgr8_to_jpeg(image)
    if state_widget.value != 'live':
        car.throttle = 0.0
        return
    debug_image, steering, info = detector.process_frame(image, draw_debug=True)
    debug_widget.value = bgr8_to_jpeg(debug_image)
    # Fail-safe: never drive forward if YOLO lost the apex.
    if not info['lane_confident']:
        car.throttle = 0.0
        status_widget.value = '<b>YOLO:</b> apex lost — stopped'
        return
    command = max(-1.0, min(1.0, steering * steering_gain_slider.value))
    car.steering = command
    car.throttle = throttle_slider.value
    status_widget.value = '<b>YOLO:</b> apex=({:.0f}, {:.0f}), conf={:.2f}, steering={:.2f}'.format(info['apex_x'], info['apex_y'], info['confidence'], command)

camera.observe(live_update, names='value')
camera.running = True
print('Camera ready. Set State = live when the car is safe to move.')

In [ ]:
# Emergency stop / clean shutdown
state_widget.value = 'stop'
car.throttle = 0.0
camera.running = False
camera.unobserve_all()